# 第6讲：视频-音频融合与智能归档
## 用 Whisper + LLaMA-3 构建完整的「音频→文本→摘要」归档流水线

> **本讲定位**：工具链集成讲——上承第5讲视频帧处理，下接第1讲文本摘要，补齐视频内容理解三件套的最后一块拼图（画面 + 声音 + 文字）。

# 本讲定位

第1讲我们学习了如何对文本做摘要（当时使用的是模拟文本）。第5讲我们用 BLIP-2 对视频帧生成了画面描述。

然而，数字人文归档中还有一个关键环节尚未打通：**如何从视频中自动提取音频，并将其转为可检索、可摘要的文字转写？**

本讲引入 Whisper——目前最成熟的开源语音识别模型：
- 完全离线运行，无需联网或 API 密钥
- 支持 99 种语言，中文识别词错误率约 9%（Small 模型，参考值）
- 模型仅约 500 MB，Colab 免费 T4 可流畅运行

更重要的是，本讲将 Whisper 的转写输出**直接接入第1讲的 LLaMA-3 摘要流水线**，完成「视频音频 → 文本转写 → 归档摘要」的全链路闭环。

# 本讲结构

## 1  环境准备与 GPU 自检
## 2  Whisper 模型基础架构（简化版）
## 3  加载 Whisper（Small 模型）
## 4  检查点：确认模型可转写音频
## 5  从视频中提取音频（ffmpeg）
## 6  转写音频 → 文本（含时间戳）
## 7  联动 LLaMA-3 生成归档摘要（复用第1讲）
## 8  结果展示与质量评估
## 快速回顾

# 透明度说明

> **模型来源**：本讲使用 `openai/whisper-small`（通过 `openai-whisper` Python 包加载），这是 OpenAI 于 2022 年发布的开源语音识别模型，基于 MIT 许可证，可从 Hugging Face 直接下载，无需 API 密钥或联网调用。LLaMA-3 部分复用第1讲的 `NousResearch/Meta-Llama-3-8B-Instruct`。
>
> **架构简述**：Whisper 采用 Encoder-Decoder 架构——编码器将音频波形转为对数梅尔频谱图，通过 CNN + Transformer 提取声学特征；解码器以自回归方式逐 token 生成文本转写。同一模型支持多语言识别（transcribe）和翻译（translate）两种任务。
>
> **适用边界**：Whisper 擅长清晰的语音（访谈、讲座、旁白），但对强背景噪声、方言口音、多人重叠语音的识别精度明显下降。对于低质量田野录音，建议先用音频增强工具预处理，或升级至 Large-v3 模型。
>
> **数据伦理提示**：当转写真实采访、口述史等音频时，请注意受访者的知情同意——语音数据属于生物特征信息，受《个人信息保护法》保护。

# 第1章  环境准备与 GPU 自检

- 安装核心依赖：`openai-whisper`（语音识别模型）、`torch`（深度学习框架）、`soundfile`（WAV 读写）
- Colab 默认已安装 ffmpeg；本地用户若未安装，Whisper 首次运行时会自动提示

> **本地用户**：如果已在 `mllm-dh` conda 环境中运行，可注释掉 `!pip install` 行，直接执行 GPU 自检。
>
> **注意**：Whisper 在 CPU 上也可运行（Small 模型约 3-5 秒处理 1 秒音频），GPU 上约 0.3-0.5 秒/秒音频。建议使用 GPU。

In [ ]:
# ===== 第1章：环境准备 =====
# Colab 用户：运行此单元格自动安装依赖。
# 本地 mllm-dh 环境用户：可注释掉 !pip 行，直接运行 GPU 自检。

!pip install -q openai-whisper torch soundfile

import torch
import whisper

if not torch.cuda.is_available():
    print("未检测到 GPU。Whisper 可在 CPU 上运行，但速度较慢（约 3-5 秒/秒音频）。")
    print("Colab 用户请选择：运行时 -> 更改运行时类型 -> T4 GPU。")
else:
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU 已就绪：{device_name}")
    print(f"显存总量：{total_mem:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备：{DEVICE}")

# 第2章  Whisper 模型基础架构（简化版）

## Whisper 如何将语音转为文字？

Whisper 是一个 **Encoder-Decoder** 架构的语音识别模型，工作流程如下：

| 阶段 | 组件 | 作用 |
|------|------|------|
| 音频预处理 | 对数梅尔频谱图 | 将音频波形转为二维「图像」（时间 × 频率） |
| 编码 | CNN + Transformer Encoder | 从频谱图中提取声学特征，捕捉发音、语调信息 |
| 解码 | Transformer Decoder | 以自回归方式逐 token 生成文本，每个新 token 基于音频特征 + 已生成的文本 |

关键特性：
- **多语言**：训练数据含 99 种语言，输入音频后自动检测语言
- **多任务**：同一模型可做转写（transcribe，保留原始语言）或翻译（translate，非英语 → 英语）
- **时间戳**：除转写文本外，还可输出每个词的时间起止点，便于与视频帧对齐

# 第3章  加载 Whisper（Small 模型）

## 为什么选 Small 而非 Large？

Whisper 有多个尺寸，本讲选择 **Small**：

| 模型 | 参数量 | 显存需求 | 相对速度 | 中文 WER（参考） |
|------|--------|---------|---------|------------------|
| tiny | 39M | ~1 GB | 10× | ~15% |
| base | 74M | ~1 GB | 7× | ~12% |
| **small** | **244M** | **~2 GB** | **4×** | **~9%** |
| medium | 769M | ~5 GB | 2× | ~7% |
| large-v3 | 1.5B | ~10 GB | 1× | ~5% |

Small 模型在精度和速度之间取得最佳平衡：中文识别词错误率约 9%，T4 GPU 上处理 1 分钟音频仅需约 20 秒。

> **首次加载**：`whisper.load_model("small")` 会自动从 Hugging Face 下载约 500 MB 权重，Colab 约需 30 秒。

In [ ]:
# ===== 第3章：加载 Whisper 模型 =====
# 首次运行需下载约 500 MB 权重，Colab 大约需要 30 秒

MODEL_SIZE = "small"

whisper_model = whisper.load_model(MODEL_SIZE, device=DEVICE)
print(f"Whisper {MODEL_SIZE} 模型加载完成，设备：{DEVICE}")

# 第4章  检查点：确认模型可转写音频

在进行视频音频提取之前，先用一段**合成音频**验证 Whisper 模型加载正确、可正常推理。

这是一个**有意设置的检查点**：如果模型权重损坏、显存不足或依赖缺失，这里会立即报错——避免在提取了大量音频后才发现模型不可用。

In [ ]:
# ===== 检查点：用合成音频验证 Whisper =====
import numpy as np
import soundfile as sf

# 生成一段简单的正弦波音频（440 Hz，持续 2 秒，16 kHz 采样率）
# 这不是语音，无法给出有意义的转写——仅用于验证模型推理不报错
sample_rate = 16000
duration = 2.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
audio_signal = 0.3 * np.sin(2 * np.pi * 440 * t)
audio_signal = audio_signal.astype(np.float32)

# 检查点：调用 Whisper 推理
print("正在用合成音频验证模型推理...")
result = whisper_model.transcribe(audio_signal)
print(f"合成音频转写结果：\"{result['text'][:80]}\"")
print("（结果可能为空或无语义——这是正常的，因为输入不是语音）")
print("=" * 20)
print("Whisper 模型加载正常，推理流程通畅，可以继续。")

# 第5章  从视频中提取音频（ffmpeg）

## 音频提取策略

Whisper 期望输入为 16 kHz 单声道音频（NumPy 数组或文件路径）。本讲采用：
1. 使用 ffmpeg（`subprocess` 调用）从视频中提取音频轨
2. 重采样至 16 kHz 单声道，16 位 PCM 格式
3. 读取为 NumPy 数组供 Whisper 处理

> **提示**：如果本地已有 mp4 视频，将 `VIDEO_PATH` 改为你的文件路径。Colab 用户请先上传视频。
>
> **复用说明**：本讲的音频提取可接入第5讲的视频帧提取流水线，实现视频 → 帧（L05）+ 视频 → 音频（L06）的并行处理。

In [ ]:
# ===== 第5章：从视频中提取音频 =====
import subprocess
import os

# --- 配置区 ---
VIDEO_PATH = None  # 设为 None 则自动查找当前目录的 mp4/mov 文件

# 自动查找视频文件
IS_REAL_VIDEO = False
if VIDEO_PATH is None:
    import glob
    candidates = glob.glob("*.mp4") + glob.glob("*.mov") + glob.glob("*.mkv")
    if candidates:
        VIDEO_PATH = candidates[0]
        print(f"自动选择视频：{VIDEO_PATH}")
        IS_REAL_VIDEO = True
    else:
        print("未找到视频文件，将使用测试音频占位。")
        print("如果你已上传视频，请将 VIDEO_PATH 设为完整路径。")
else:
    IS_REAL_VIDEO = True


def extract_audio(video_path):
    """用 ffmpeg 提取 16kHz 单声道 WAV，返回 NumPy 数组和采样率。"""
    output_audio = "temp_audio.wav"
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vn",               # 丢弃视频轨
        "-acodec", "pcm_s16le",  # 16位 PCM
        "-ar", "16000",      # 16 kHz
        "-ac", "1",          # 单声道
        "-y",                # 覆盖已有文件
        output_audio,
    ]
    try:
        subprocess.run(cmd, capture_output=True, check=True)
        audio_data, sr = sf.read(output_audio)
        return audio_data.astype(np.float32), sr
    finally:
        if os.path.exists(output_audio):
            os.remove(output_audio)  # 保证无论是否成功都清理


def _make_placeholder_audio(duration=5, freq=220, sr=16000):
    """生成占位正弦波音频（无语音内容），用于无视频时的演示。"""
    n_samples = int(sr * duration)
    t = np.linspace(0, duration, n_samples, endpoint=False)
    audio = (0.3 * np.sin(2 * np.pi * freq * t)).astype(np.float32)
    return audio, sr


if IS_REAL_VIDEO and os.path.exists(VIDEO_PATH):
    try:
        audio, sr = extract_audio(VIDEO_PATH)
        duration_sec = len(audio) / sr
        print(f"真实视频已加载：{VIDEO_PATH}")
        print(f"采样率：{sr} Hz")
        print(f"时长：{duration_sec:.1f} 秒（{duration_sec/60:.1f} 分钟）")
    except subprocess.CalledProcessError as e:
        print(f"ffmpeg 提取失败：{e.stderr.decode() if e.stderr else e}")
        print("回退至测试音频占位。后续转写结果并非真实视频内容！")
        IS_REAL_VIDEO = False
        audio, sr = _make_placeholder_audio()
    except Exception as e:
        print(f"音频提取失败：{e}")
        print("回退至测试音频占位。后续转写结果并非真实视频内容！")
        IS_REAL_VIDEO = False
        audio, sr = _make_placeholder_audio()
else:
    print("当前使用测试音频占位——后续转写结果并非真实视频内容！")
    audio, sr = _make_placeholder_audio()
    print(f"测试音频时长：{len(audio)/sr:.1f} 秒，采样率：{sr} Hz")

print(f"音频数组形状：{audio.shape}")
if not IS_REAL_VIDEO:
    print("WARNING:  当前使用占位音频，后续转写/语言检测结果无实际意义。")

# 第6章  转写音频 → 文本

## Whisper 转写参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `language` | 自动检测 | 可手动指定（如 `"zh"`、`"en"`），跳过语言检测加快速度 |
| `task` | `"transcribe"` | 转写（保留原始语言）或 `"translate"`（翻译为英文） |
| `beam_size` | 5 | 束搜索宽度——值越大越精准但越慢 |
| `word_timestamps` | False | 是否输出逐词时间戳（用于视频帧对齐） |

本讲使用**转写模式**：自动检测语言，输出原始语言文本。中文短视频的转写结果可直接用于后续检索与摘要。

> **提示**：Whisper 也支持翻译模式（`task="translate"`），可将任意语言的音频直接翻译为英文文本，适用于跨语言归档场景。

In [ ]:
# ===== 第6章：转写音频 → 文本 =====

if not IS_REAL_VIDEO:
    print("WARNING:  当前为占位音频，以下转写结果无实际语义。\n")

# 转写（含时间戳，便于后续与视频帧对齐）
print(f"正在转写音频（模型：{MODEL_SIZE}，设备：{DEVICE}）...")

result = whisper_model.transcribe(
    audio,
    task="transcribe",       # 转写（保留原始语言）
    beam_size=5,
)

detected_lang = result["language"]
full_text = result["text"].strip()

print(f"检测语言：{detected_lang}")
print(f"全文长度：{len(full_text)} 字符")
print()

# 打印片段（每段时间戳 + 文本）
segments = result["segments"]
print(f"片段数量：{len(segments)}")
print()
for seg in segments:
    start = f"{seg['start']:.1f}s"
    end = f"{seg['end']:.1f}s"
    text = seg["text"].strip()
    if text:
        print(f"  [{start} -> {end}]  {text[:80]}{'...' if len(text) > 80 else ''}")

print()
print("--- 完整转写文本 ---")
print(full_text)

# 第7章  联动 LLaMA-3 生成归档摘要（复用第1讲）

## 从转写到归档的闭环

以下是 Whisper 转写与第1讲摘要流水线的**完整对接**：

```
视频 (mp4)
  ├── Whisper 提取音频 → 转写文本     [本讲 L06]
  └── LLaMA-3 读入转写文本 → 生成摘要  [第1讲 L01]
```

> **模板说明**：以下代码复用第1讲的 LLaMA-3 加载与摘要生成逻辑。如果你的环境中尚未加载 LLaMA-3，先运行第1讲中加载 `model` 和 `tokenizer` 的单元格（使用 `NousResearch/Meta-Llama-3-8B-Instruct`），确保两个变量已存在于当前内核中。

In [ ]:
# ===== 第7章：联动 LLaMA-3 摘要（需先运行 L01 的模型加载代码）=====

# 本单元格依赖第1讲的 LLaMA-3 模型和分词器。
# 确保变量 model 和 tokenizer（LLaMA-3）已从前面单元格加载。
# 若未加载，先执行第1讲中加载 LLaMA-3 的单元格。

if not IS_REAL_VIDEO:
    print("WARNING:  当前为占位音频，以下摘要无实际意义。\n")

USE_LLAMA = False

try:
    _ = tokenizer
    _ = model
    USE_LLAMA = True
except NameError:
    print("未检测到 LLaMA-3。请先运行第1讲中加载 model 和 tokenizer 的单元格。")
    print("本单元格将跳过摘要生成，仅展示转写文本。")

if USE_LLAMA:
    # prompt 可按视频领域自行修改
    SYSTEM_PROMPT = (
        "你是一位专业的归档员，擅长为视频转写文本撰写精炼的归档摘要。"
        "摘要要求：100字以内，保留关键术语，结构清晰，适合文献归档。"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为以下短视频转写文本生成一份100字以内的归档摘要：{full_text}"},
    ]

    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        summary = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        print("===== LLaMA-3 生成摘要 =====")
        print(summary)
        print("=" * 28)
        print(f"摘要长度：{len(summary)} 字符")
        print(f"转写原文长度：{len(full_text)} 字符")
        print(f"字符级压缩比：{len(summary) / max(len(full_text), 1) * 100:.1f}%")
    except Exception as e:
        print(f"LLaMA-3 推理失败：{e}")
        print("请检查 model / tokenizer 是否来自第1讲的 LLaMA-3 加载单元格。")
        summary = None
else:
    summary = None
    print("（已跳过 LLaMA-3 摘要，仅展示转写文本。）")

# 第8章  结果展示与质量评估

## Whisper 转写质量的参考维度

| 维度 | 检查要点 |
|------|----------|
| **语言检测** | 是否正确识别语言？中英混读时 Whisper 可能只标记一种主语言 |
| **专有名词** | 人名、地名、古籍术语是否被错误转写为同音词？ |
| **断句合理** | Whisper 的自动分句是否与说话人的自然停顿一致？ |
| **噪声容忍** | 背景音较大时，是否出现漏转或乱码段？ |

> **已知局限**：Whisper Small 模型对专业术语（如「缮写」→「山写」）和方言口音（如吴语、闽语）的识别精度有限。对于高精度的学术转写需求（如古籍口述史整理），建议升级至 Large-v3 模型或使用讯飞等专业 ASR 服务。

In [ ]:
# ===== 第8章：归档与结果展示 =====
import json
from datetime import datetime

# 构建归档记录
archive_record = {
    "video_source": VIDEO_PATH,          # 始终保留原始路径（可能为 None）
    "is_real_video": IS_REAL_VIDEO,
    "processed_at": datetime.now().isoformat(),
    "whisper_model": f"openai/whisper-{MODEL_SIZE}",
    "detected_language": detected_lang,
    "audio_duration_seconds": len(audio) / sr,
    "transcript": full_text,
    "segments": [
        {
            "start": seg["start"],
            "end": seg["end"],
            "text": seg["text"].strip(),
        }
        for seg in segments
    ],
    "llama3_summary": summary if USE_LLAMA else None,
}

# 保存归档
OUTPUT_PATH = "L06_archive_output.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(archive_record, f, ensure_ascii=False, indent=2)

print(f"归档记录已保存至：{OUTPUT_PATH}")
print()
print("--- 归档内容概要 ---")
print(f"检测语言：{archive_record['detected_language']}")
print(f"音频时长：{archive_record['audio_duration_seconds']:.1f} 秒")
print(f"转写片段数：{len(archive_record['segments'])}")
print(f"转写字符数：{len(archive_record['transcript'])}")
if archive_record["llama3_summary"]:
    print(f"LLaMA-3 摘要：{archive_record['llama3_summary'][:100]}...")

print()
print("提示：可通过比对转写文本与原音频来评估 Whisper 的识别精度，参考上方质量评估维度。")
print("TIP:  提示：Colab 运行时重置后文件会丢失，请及时下载 JSON 或挂载 Google Drive 保存。")

# 快速回顾（Quick Check）

**本讲完成了以下工作：**

- **Whisper 加载**：使用 `openai/whisper-small`（约 500 MB 权重），T4 GPU 可流畅运行
- **音频提取**：用 ffmpeg 从视频中提取 16 kHz 单声道音频
- **语音转写**：Whisper 自动检测语言并逐段转写，输出带时间戳的文本
- **流水线对接**：将转写文本直接接入第1讲的 LLaMA-3 摘要流程，实现「音频 → 文本 → 摘要」闭环
- **归档输出**：结构化 JSON 记录包含转写全文、逐段时间戳及可选摘要

> **和前后各讲的连贯性**：L01 提供文本摘要能力，L05 提供视频帧描述能力。**L06 补齐了音频 → 文本这一关键环节**，至此构成完整的视频内容理解三件套（画面 + 声音 + 文字）。L07 的 LLaVA-Next 将在此基础上实现原生多语言视觉问答。
>
> **本讲对后续的依赖**：转写文本可直接作为 L03（QLoRA 微调）和 L04 的训练语料，用于构建面向数字人文短视频的领域专用知识库。